# STCP - Modelo Matemático do Algoritmo de Decisão de Carregamento

Este notebook documenta formalmente as fórmulas matemáticas do sistema de decisão de carregamento de autocarros eléctricos da STCP.

O sistema responde à pergunta central: **dado o estado actual de toda a frota e de todas as estações, qual o autocarro que deve carregar a seguir, em que estação, e com que urgência?**

O modelo integra quatro subsistemas interdependentes:
- **Frota** — estado individual e agregado de todos os autocarros
- **Estações de Carregamento** — disponibilidade, capacidade e saúde dos carregadores
- **Serviços e Rotas** — dados GTFS reais das linhas operacionais
- **Prioridades** — sistema de escalonamento que combina todos os factores

**Autor:** Luís Pereira  
**Instituição:** ISCAP  
**Data:** Junho 2026

## 1. Parâmetros e Constantes do Sistema

### 1.1 Parâmetros do Autocarro (CaetanoBus e.City Gold)

| Símbolo | Descrição | Valor | Unidade |
|---|---|---|---|
| $C_{bat}$ | Capacidade total da bateria | 350 | kWh |
| $c_{km}$ | Consumo médio por quilómetro | 1.2 | kWh/km |
| $SOC_{min}$ | Nível mínimo absoluto de segurança | 20 | % |
| $SOC_{max}$ | Nível máximo de carregamento | 95 | % |
| $SOC_{ideal\_min}$ | Início da faixa operacional ideal | 25 | % |
| $SOC_{ideal\_max}$ | Fim da faixa operacional ideal | 85 | % |
| $T_{bat\_max}$ | Temperatura máxima da bateria | 45 | °C |
| $T_{bat\_ideal\_min}$ | Temperatura ideal mínima | 15 | °C |
| $T_{bat\_ideal\_max}$ | Temperatura ideal máxima | 35 | °C |

### 1.2 Parâmetros das Estações de Carregamento

| Símbolo | Descrição | Valor | Unidade |
|---|---|---|---|
| $N_{cs}$ | Número total de estações | 3 | - |
| $P_{max}$ | Potência máxima por estação | 120 | kW |
| $P_{rede}$ | Limite total da rede eléctrica | 300 | kW |
| $\eta$ | Eficiência do carregamento | 0.92 | - |
| $T_{cs\_max}$ | Temperatura máxima do carregador | 60 | °C |

### 1.3 Parâmetros do Sistema

| Símbolo | Descrição | Valor | Unidade |
|---|---|---|---|
| $N_{bus}$ | Número total de autocarros na frota | 3 | - |
| $m$ | Margem de segurança de energia | 1.20 | - |
| $\varphi_{depot}$ | Latitude do depósito STCP Bonjóia | 41.152661 | ° |
| $\lambda_{depot}$ | Longitude do depósito STCP Bonjóia | -8.579658 | ° |
| $R$ | Raio médio da Terra | 6371 | km |

## 2. Variáveis de Estado em Tempo Real

### 2.1 Estado Individual de Cada Autocarro

Para cada autocarro $i \in \{1, ..., N_{bus}\}$ no instante $t$:

| Símbolo | Descrição | Fonte | Unidade |
|---|---|---|---|
| $SOC_i(t)$ | Estado de carga actual | Telemetria | % |
| $E_i(t)$ | Energia actual na bateria | Telemetria | kWh |
| $T_i(t)$ | Temperatura actual da bateria | Telemetria | °C |
| $\varphi_i(t)$ | Latitude actual | Telemetria | ° |
| $\lambda_i(t)$ | Longitude actual | Telemetria | ° |
| $d_i(t)$ | Distância percorrida na rota actual | Telemetria | m |
| $s_i(t)$ | Estado operacional | Telemetria | {parked, charging, inTransit} |
| $r_i$ | Rota atribuída | Configuração | {200, 201, 202} |

### 2.2 Estado Individual de Cada Estação

Para cada estação $j \in \{1, ..., N_{cs}\}$ no instante $t$:

| Símbolo | Descrição | Fonte | Unidade |
|---|---|---|---|
| $e_j(t)$ | Estado da estação | Telemetria | {available, occupied, fault} |
| $P_j(t)$ | Potência actual entregue | Telemetria | kW |
| $T_j(t)$ | Temperatura do carregador | Telemetria | °C |
| $B_j(t)$ | Autocarro actualmente ligado | Telemetria | ID ou null |
| $E_{delivered,j}(t)$ | Energia entregue na sessão actual | Telemetria | kWh |

### 2.3 Estado Agregado da Frota

O estado agregado da frota é calculado a partir dos estados individuais e é essencial para as decisões de priorização:

$$N_{transit}(t) = \sum_{i} \mathbf{1}[s_i(t) = \text{inTransit}]$$

$$N_{parked}(t) = \sum_{i} \mathbf{1}[s_i(t) = \text{parked}]$$

$$N_{charging}(t) = \sum_{i} \mathbf{1}[s_i(t) = \text{charging}]$$

Verificação: $N_{transit}(t) + N_{parked}(t) + N_{charging}(t) = N_{bus}$

### 2.4 Estado Agregado das Estações

$$N_{cs\_avail}(t) = \sum_{j} \mathbf{1}[e_j(t) = \text{available} \text{ e } T_j(t) < T_{cs\_max}]$$

$$N_{cs\_fault}(t) = \sum_{j} \mathbf{1}[e_j(t) = \text{fault}]$$

$$P_{rede\_actual}(t) = \sum_{j} P_j(t)$$

### 2.5 Indicador de Pressão sobre a Rede de Carregamento

O indicador de pressão mede a relação entre a procura de carregamento e a capacidade disponível:

$$N_{deficit}(t) = \sum_{i} \mathbf{1}[\Delta SOC_i(t) > 0]$$

$$\Pi(t) = \frac{N_{deficit}(t)}{\max(N_{cs\_avail}(t), 1)}$$

Onde $\Pi(t) > 1$ indica que há mais autocarros em défice do que estações disponíveis — situação crítica que exige priorização rigorosa.

## 3. Modelo das Rotas (Dados GTFS)

### 3.1 Parâmetros das Rotas

Os dados das rotas provêm dos ficheiros GTFS reais da STCP Porto, armazenados na tabela Delta `route_stops`:

| Rota $r$ | Autocarro | Distância Total $D_{total,r}$ | Paragens $N_{stops,r}$ | Prioridade |
|---|---|---|---|---|
| Linha 200 | BUS-001 | 9853.9 m | 30 | Alta |
| Linha 201 | BUS-002 | 9777.6 m | 26 | Normal |
| Linha 202 | BUS-003 | 5313.2 m | 16 | Crítica |

### 3.2 Distância Restante na Rota

$$D_{rota,i}(t) = D_{total,r_i} - d_i(t) \quad [\text{m}]$$

### 3.3 Paragem Actual na Rota

A paragem actual é determinada cruzando $d_i(t)$ com as distâncias cumulativas das paragens na tabela `route_stops`:

$$k_i(t) = \max\left\{k : dist\_cumul_k \leq d_i(t)\right\}$$

Onde $k_i(t)$ é o índice da última paragem ultrapassada pelo autocarro $i$.

### 3.4 Paragens Restantes

$$N_{stops\_restantes,i}(t) = N_{stops,r_i} - k_i(t)$$

### 3.5 Previsão de Chegada ao Depósito

Estimativa do tempo restante até ao autocarro $i$ completar a rota e regressar ao depósito:

$$t_{chegada,i}(t) = \frac{D_{rota,i}(t) + D_i(t) \times 1000}{v_{media}} \quad [\text{segundos}]$$

Onde $v_{media} = 20$ km/h $= 5.56$ m/s é a velocidade média urbana.

Esta estimativa permite antecipar necessidades de carregamento mesmo antes de o autocarro entrar em défice.

## 4. Cálculo da Distância ao Depósito

### 4.1 Fórmula de Haversine (Exacta)

A distância geodésica entre o autocarro $i$ e o depósito:

$$\Delta\varphi = (\varphi_{depot} - \varphi_i(t)) \cdot \frac{\pi}{180}$$

$$\Delta\lambda = (\lambda_{depot} - \lambda_i(t)) \cdot \frac{\pi}{180}$$

$$a = \sin^2\left(\frac{\Delta\varphi}{2}\right) + \cos\left(\varphi_i \cdot \frac{\pi}{180}\right) \cdot \cos\left(\varphi_{depot} \cdot \frac{\pi}{180}\right) \cdot \sin^2\left(\frac{\Delta\lambda}{2}\right)$$

$$D_i(t) = 2R \cdot \arctan2\left(\sqrt{a},\ \sqrt{1-a}\right) \quad [\text{km}]$$

### 4.2 Aproximação Plana (Implementação SQL)

Válida para distâncias inferiores a 50 km, como no caso do Porto:

$$D_i(t) \approx 111.0 \cdot \sqrt{(\varphi_{depot} - \varphi_i)^2 + (\lambda_{depot} - \lambda_i)^2 \cdot \cos^2\left(\varphi_i \cdot \frac{\pi}{180}\right)} \quad [\text{km}]$$

O erro desta aproximação é inferior a 0.5% para as distâncias do Porto, sendo aceitável para decisões operacionais.

## 5. Modelo de Energia

### 5.1 Energia para Completar a Rota Actual

$$E_{rota,i}(t) = \frac{D_{rota,i}(t)}{1000} \cdot c_{km} \quad [\text{kWh}]$$

$$SOC_{rota,i}(t) = \frac{E_{rota,i}(t)}{C_{bat}} \cdot 100 \quad [\%]$$

### 5.2 Energia para Regressar ao Depósito

$$E_{volta,i}(t) = D_i(t) \cdot c_{km} \cdot m \quad [\text{kWh}]$$

$$SOC_{volta,i}(t) = \frac{E_{volta,i}(t)}{C_{bat}} \cdot 100 \quad [\%]$$

### 5.3 Bateria Total Necessária

A bateria total necessária para o autocarro $i$ completar a rota actual E regressar ao depósito com margem de segurança:

$$SOC_{necesario,i}(t) = SOC_{rota,i}(t) + SOC_{volta,i}(t) + SOC_{min}$$

### 5.4 Défice de Bateria

$$\Delta SOC_i(t) = SOC_{necesario,i}(t) - SOC_i(t) \quad [\%]$$

Se $\Delta SOC_i(t) > 0$, o autocarro está em défice e deve carregar.

### 5.5 Consumo Previsto até ao Fim da Rota

Previsão do nível de bateria quando o autocarro chegar ao depósito, sem carregamento:

$$SOC_{previsto,i}(t) = SOC_i(t) - SOC_{rota,i}(t) - SOC_{volta,i}(t)$$

Se $SOC_{previsto,i}(t) < SOC_{min}$, o autocarro não consegue completar o percurso completo de forma segura.

## 6. Restrições de Segurança

### 6.1 Restrições do Autocarro

**R1 - Temperatura da bateria:**
$$T_i(t) \leq T_{bat\_max} = 45°C$$

**R2 - Nível máximo de bateria:**
$$SOC_i(t) < SOC_{max} = 95\%$$

**R3 - Défice positivo (necessidade real):**
$$\Delta SOC_i(t) > 0$$

### 6.2 Restrições da Estação

**R4 - Disponibilidade:**
$$e_j(t) = \text{available}$$

**R5 - Temperatura do carregador:**
$$T_j(t) < T_{cs\_max} = 60°C$$

**R6 - Capacidade da rede eléctrica:**
$$P_{rede\_actual}(t) + P_{final,i} \leq P_{rede} = 300 \text{ kW}$$

### 6.3 Função de Elegibilidade do Autocarro

$$\text{Elegível}_i(t) = \mathbf{1}[R1 \text{ e } R2 \text{ e } R3]$$

### 6.4 Função de Disponibilidade da Estação

$$\text{Disponível}_j(t) = \mathbf{1}[R4 \text{ e } R5]$$

## 7. Sistema de Priorização

O score de urgência $U_i(t)$ determina a ordem de alocação às estações. É composto por quatro componentes que reflectem diferentes dimensões do problema:

### 7.1 Componente de Prioridade do Serviço ($P_{serv}$)

Reflecte a importância operacional da linha:

$$P_{serv,i} = \begin{cases} 40 & \text{se prioridade}(r_i) = \text{Crítica} \\ 30 & \text{se prioridade}(r_i) = \text{Alta} \\ 20 & \text{se prioridade}(r_i) = \text{Normal} \end{cases}$$

### 7.2 Componente de Défice de Bateria ($P_{bat}$)

Reflecte a gravidade do défice energético:

$$P_{bat,i} = \begin{cases} 60 & \text{se } \Delta SOC_i > 0 \quad \text{(em défice)} \\ 30 & \text{se } \Delta SOC_i > -10\% \quad \text{(margem reduzida)} \\ 0 & \text{caso contrário} \end{cases}$$

### 7.3 Componente de Distância ao Depósito ($P_{dist}$)

Reflecte o risco de o autocarro não conseguir regressar:

$$P_{dist,i} = \begin{cases} 100 & \text{se } SOC_i(t) < SOC_{volta,i}(t) \quad \text{(não consegue regressar)} \\ 60 & \text{se } SOC_i(t) < SOC_{volta,i}(t) + 5\% \quad \text{(margem crítica)} \\ 0 & \text{caso contrário} \end{cases}$$

### 7.4 Componente de Pressão da Frota ($P_{frota}$)

Reflecte o contexto global da frota — quando há mais autocarros em défice do que estações disponíveis, os autocarros parados têm menor prioridade do que os que estão em rota:

$$P_{frota,i} = \begin{cases} 20 & \text{se } \Pi(t) > 1 \text{ e } s_i(t) = \text{inTransit} \\ 10 & \text{se } \Pi(t) > 1 \text{ e } s_i(t) = \text{parked} \\ 0 & \text{se } \Pi(t) \leq 1 \end{cases}$$

### 7.5 Score Total de Urgência

$$U_i(t) = P_{serv,i} + P_{bat,i} + P_{dist,i} + P_{frota,i}$$

Os autocarros elegíveis são ordenados de forma decrescente por $U_i(t)$.

### 7.6 Intervalo Teórico do Score

| Componente | Mínimo | Máximo |
|---|---|---|
| $P_{serv}$ | 20 | 40 |
| $P_{bat}$ | 0 | 60 |
| $P_{dist}$ | 0 | 100 |
| $P_{frota}$ | 0 | 20 |
| **Total** | **20** | **220** |

## 8. Modelo das Estações de Carregamento

### 8.1 Classificação do Estado de Cada Estação

$$\text{Estado}_j(t) = \begin{cases} \text{DISPONÍVEL} & \text{se } e_j = \text{available} \text{ e } T_j < 60°C \\ \text{OCUPADA} & \text{se } e_j = \text{occupied} \\ \text{AVARIA} & \text{se } e_j = \text{fault} \\ \text{SOBREAQUECIDA} & \text{se } T_j \geq 60°C \end{cases}$$

### 8.2 Capacidade Disponível na Rede

$$P_{disponivel}(t) = P_{rede} - P_{rede\_actual}(t) = 300 - \sum_{j} P_j(t) \quad [\text{kW}]$$

### 8.3 Tempo Estimado de Libertação de uma Estação Ocupada

Para uma estação $j$ ocupada a carregar o autocarro $k$:

$$SOC_{alvo,k} = SOC_{necesario,k}(t) + 5\%$$

$$\Delta E_k = \frac{(SOC_{alvo,k} - SOC_k(t))}{100} \cdot C_{bat} \quad [\text{kWh}]$$

$$t_{libertacao,j}(t) = \frac{\Delta E_k}{P_j(t) \cdot \eta} \cdot 60 \quad [\text{minutos}]$$

Esta informação é usada para decidir se um autocarro deve aguardar ou procurar outra estação.

## 9. Ajuste Dinâmico da Potência de Carregamento

A potência de carregamento não é fixa — é ajustada dinamicamente com base em três factores:

### 9.1 Ajuste por Temperatura da Bateria do Autocarro

$$P_{temp,i} = \begin{cases} 0 \text{ kW} & \text{se } T_i > 45°C \quad \text{(BLOQUEADO — risco de degradação)} \\ 80 \text{ kW} & \text{se } 35°C < T_i \leq 45°C \quad \text{(REDUZIDO)} \\ 50 \text{ kW} & \text{se } T_i < 15°C \quad \text{(PRÉ-AQUECIMENTO)} \\ P_{max} & \text{se } 15°C \leq T_i \leq 35°C \quad \text{(NORMAL)} \end{cases}$$

### 9.2 Ajuste por Balanceamento da Rede Eléctrica

$$f_{bal}(t) = \begin{cases} 0.80 & \text{se } P_{rede\_actual}(t) > 0.90 \cdot P_{rede} \\ 0.90 & \text{se } P_{rede\_actual}(t) > 0.80 \cdot P_{rede} \\ 1.00 & \text{caso contrário} \end{cases}$$

### 9.3 Ajuste por Urgência Relativa

Quando a rede está sob pressão ($\Pi(t) > 1$), os autocarros com score mais alto recebem maior potência:

$$f_{urgencia,i}(t) = \begin{cases} 1.00 & \text{se } U_i(t) \geq 150 \quad \text{(urgência máxima)} \\ 0.90 & \text{se } 100 \leq U_i(t) < 150 \\ 0.80 & \text{se } U_i(t) < 100 \end{cases}$$

Este factor só é aplicado quando $\Pi(t) > 1$, ou seja, quando há mais autocarros em défice do que estações.

### 9.4 Potência Final

$$P_{final,i}(t) = \min\left(P_{max},\ P_{disponivel}(t),\ P_{temp,i} \cdot f_{bal}(t) \cdot f_{urgencia,i}(t)\right) \quad [\text{kW}]$$

A restrição $P_{disponivel}(t)$ garante que a potência alocada não excede o limite da rede eléctrica.

## 10. Tempo Estimado de Carregamento

### 10.1 Nível Alvo de Carregamento

O objectivo não é carregar até ao máximo, mas apenas até ao mínimo necessário para o próximo ciclo operacional:

$$SOC_{alvo,i}(t) = \min\left(SOC_{ideal\_max},\ SOC_{necesario,i}(t) + 5\%\right)$$

### 10.2 Energia a Carregar

$$\Delta E_i(t) = \frac{\max(SOC_{alvo,i}(t) - SOC_i(t), 0)}{100} \cdot C_{bat} \quad [\text{kWh}]$$

### 10.3 Tempo Estimado

$$t_{carga,i}(t) = \frac{\Delta E_i(t)}{P_{final,i}(t) \cdot \eta} \cdot 60 \quad [\text{minutos}]$$

### 10.4 Exemplo Numérico — BUS-003

- $SOC_i = 25\%$, $D_{rota,i} = 200$ m, $D_i = 8$ km
- $SOC_{rota} = (0.2/1000 \times 1.2 / 350) \times 100 = 0.07\%$
- $SOC_{volta} = (8 \times 1.2 \times 1.2 / 350) \times 100 = 3.29\%$
- $SOC_{necesario} = 0.07 + 3.29 + 20 = 23.36\%$
- $\Delta SOC = 23.36 - 25 = -1.64\%$ → sem défice imediato
- $SOC_{previsto} = 25 - 0.07 - 3.29 = 21.64\%$ → acima de $SOC_{min}$, seguro

Se $SOC_i = 22\%$:
- $\Delta SOC = 23.36 - 22 = 1.36\%$ → em défice
- $SOC_{alvo} = 23.36 + 5 = 28.36\%$
- $\Delta E = (28.36 - 22)/100 \times 350 = 22.26$ kWh
- $t_{carga} = 22.26 / (120 \times 0.92) \times 60 = 12.1$ minutos

## 11. Algoritmo de Decisão Integrado

O algoritmo executa os seguintes passos em cada ciclo de avaliação, integrando todos os subsistemas:

```
═══════════════════════════════════════════════════════════
CICLO DE AVALIAÇÃO (executado a cada consulta)
═══════════════════════════════════════════════════════════

FASE 1 — RECOLHA DE ESTADO
  Para cada autocarro i:
    Obter SOC_i, T_i, lat_i, lon_i, d_i, s_i da telemetria
  Para cada estação j:
    Obter e_j, P_j, T_j da telemetria
  Calcular estado agregado:
    N_transit, N_parked, N_charging
    N_cs_avail, P_rede_actual, P_disponivel

FASE 2 — ANÁLISE ENERGÉTICA (para cada autocarro i)
  Calcular D_i usando Haversine
  Calcular SOC_rota_i, SOC_volta_i
  Calcular SOC_necesario_i = SOC_rota_i + SOC_volta_i + SOC_min
  Calcular delta_SOC_i = SOC_necesario_i - SOC_i
  Calcular SOC_previsto_i
  Calcular t_chegada_i (previsão de chegada ao depósito)

FASE 3 — VERIFICAÇÃO DE ELEGIBILIDADE (para cada autocarro i)
  Verificar R1: T_i <= 45°C
  Verificar R2: SOC_i < 95%
  Verificar R3: delta_SOC_i > 0
  Elegivel_i = R1 AND R2 AND R3

FASE 4 — CÁLCULO DE PRESSÃO DA FROTA
  N_deficit = número de autocarros com delta_SOC > 0
  Pi = N_deficit / max(N_cs_avail, 1)

FASE 5 — CÁLCULO DO SCORE (para cada autocarro elegível i)
  P_serv_i = f(prioridade da rota)
  P_bat_i = f(delta_SOC_i)
  P_dist_i = f(SOC_i vs SOC_volta_i)
  P_frota_i = f(Pi, s_i)
  U_i = P_serv_i + P_bat_i + P_dist_i + P_frota_i

FASE 6 — ORDENAÇÃO E ALOCAÇÃO
  Ordenar autocarros elegíveis por U_i decrescente
  Para cada autocarro elegível i, por ordem de urgência:
    Se há estação j disponível (Disponível_j = 1):
      Calcular f_bal usando P_rede_actual
      Calcular f_urgencia_i usando U_i e Pi
      Calcular P_final_i = min(P_max, P_disponivel, P_temp_i * f_bal * f_urgencia_i)
      Calcular t_carga_i
      DECISÃO: CARREGAR AGORA na estação j
      Actualizar P_rede_actual += P_final_i
      Marcar estação j como alocada
    Se não há estação disponível:
      Calcular t_libertacao das estações ocupadas
      DECISÃO: AGUARDAR (tempo estimado = min(t_libertacao_j))

FASE 7 — CLASSIFICAÇÃO DOS RESTANTES
  Para cada autocarro não elegível:
    Se delta_SOC_i <= 0 e SOC_previsto_i >= SOC_min:
      DECISÃO: SEM NECESSIDADE
    Se T_i > 45°C:
      DECISÃO: BLOQUEADO (temperatura)
    Se SOC_i >= 95%:
      DECISÃO: BATERIA CHEIA

RESULTADO: Tabela de decisões com acção, estação, potência,
           tempo estimado e razão para cada autocarro
═══════════════════════════════════════════════════════════
```

## 12. Diagrama de Fluxo de Decisão

```
┌─────────────────────────────────────────────────────────────┐
│              ESTADO EM TEMPO REAL (telemetria)              │
│  Autocarros: SOC, T, GPS, distância    Frota: N_transit,   │
│  Estações: estado, potência, temp      N_parked, N_charging │
└────────────────────────┬────────────────────────────────────┘
                         │
                         ▼
┌─────────────────────────────────────────────────────────────┐
│                  ANÁLISE ENERGÉTICA                         │
│  SOC_rota + SOC_volta + SOC_min = SOC_necesario             │
│  delta_SOC = SOC_necesario - SOC_actual                     │
└────────────────────────┬────────────────────────────────────┘
                         │
              delta_SOC > 0 ?
             /             \
           NÃO             SIM
            │               │
     SEM NECESSIDADE        ▼
                  T_bateria <= 45°C ?
                  /              \
                NÃO              SIM
                 │                │
           BLOQUEADO              ▼
                        CALCULAR SCORE
                    U = P_serv + P_bat + P_dist + P_frota
                         (considera estado da frota)
                                  │
                    ORDENAR por score decrescente
                                  │
                        Estação disponível ?
                        /              \
                      NÃO              SIM
                       │                │
                  AGUARDAR              ▼
              (t_libertacao)    AJUSTAR POTÊNCIA
                            P_final = f(T_bat, rede, urgência)
                                        │
                                 CARREGAR AGORA
                              (estação, potência, tempo)
```

## 13. Cenários de Operação

### Cenário A — Operação Normal ($\Pi \leq 1$)

Há estações suficientes para todos os autocarros em défice. Cada autocarro é alocado à estação disponível sem necessidade de competição. A potência é máxima ($f_{urgencia} = 1.0$).

### Cenário B — Pressão Moderada ($1 < \Pi \leq 2$)

Há mais autocarros em défice do que estações disponíveis. O score $P_{frota}$ entra em acção — autocarros em rota recebem prioridade sobre autocarros parados no depósito. A potência é reduzida para os de menor urgência ($f_{urgencia} = 0.80$ ou $0.90$).

### Cenário C — Pressão Crítica ($\Pi > 2$)

Situação de escassez severa. Apenas os autocarros com $U_i \geq 150$ (risco de não regressar ao depósito) recebem acesso imediato. Os restantes aguardam em fila ordenada por score.

### Cenário D — Estação com Avaria

Quando uma estação entra em estado `fault`, $N_{cs\_avail}$ diminui, $\Pi$ aumenta e o sistema reajusta automaticamente as prioridades. Se apenas uma estação estiver disponível, apenas o autocarro com maior $U_i$ carrega — os outros aguardam.

## 14. Referências

Os limiares operacionais e a estrutura do modelo são baseados na literatura académica:

- Faixa operacional ideal [25%, 85%] e minimização de custos: He et al. (2023) — *Optimal Model of Electric Bus Scheduling Based on Energy Consumption and Battery Loss*, MDPI Sustainability, doi:10.3390/su15129640

- Modelo de escalonamento com restrições de capacidade e janelas temporais: Liu et al. (2022) — *An optimal charging scheduling model and algorithm for electric buses*, ScienceDirect Energy, doi:10.1016/j.energy.2022.126230

- Degradação da bateria e perfil de carregamento não-linear: Yao et al. (2021) — *Optimal electric bus fleet scheduling considering battery degradation*, ScienceDirect Transportation Research Part C, doi:10.1016/j.trc.2021.103323

- Planeamento de infraestrutura com incerteza: Zhang et al. (2023) — *Electric bus charging facility planning with uncertainties*, ScienceDirect Transportation Research Part C, doi:10.1016/j.trc.2023.104068

- Especificações técnicas do autocarro: CaetanoBus e.City Gold — bateria 350 kWh, consumo médio 1.2 kWh/km

- Dados das rotas: GTFS STCP Porto — Linhas 200, 201, 202 (dados reais de paragens e distâncias)